# Analyse des achats — Black Friday

Ce notebook présente mon analyse du jeu de données Black Friday : chargement, contrôle qualité, nettoyage, indicateurs et démonstrations des services Azure.

Le parcours du projet est : Python → CSV nettoyé → Azure Blob Storage → Azure SQL → Power BI. Le stockage Blob et l’import SQL ont été réalisés séparément ; les cellules de connexion SQL ci-dessous vérifient la base existante.

## Utilisation

Placer ce notebook dans le dossier `notebooks` du projet et le fichier Kaggle `train.csv` dans `data/raw`. Pour Vision, placer `produit.jpg` ou `produit.webp` dans `notebooks`. La source du jeu de données est le jeu Black Friday de sdolezel sur Kaggle.

Exécuter les cellules dans l’ordre. La partie données fonctionne sans accès Azure. Les démonstrations Azure demandent des ressources actives et des clés saisies de façon masquée. La connexion SQL demande un compte Microsoft autorisé et le pilote Microsoft ODBC Driver 18 installé sur Windows.




## 1. Préparation et chargement

Pandas permet de lire et de manipuler les données. Les chemins sont calculés depuis le dossier du projet.

In [ ]:
# À exécuter une fois si les bibliothèques ne sont pas installées.
%pip install pandas openai azure-ai-textanalytics azure-cognitiveservices-vision-computervision azure-identity pyodbc Pillow


In [1]:
import pandas as pd
from pathlib import Path
from getpass import getpass

DOSSIER_COURANT = Path.cwd()
RACINE = DOSSIER_COURANT.parent if DOSSIER_COURANT.name == "notebooks" else DOSSIER_COURANT
DATA = RACINE / "data" / "raw" / "train.csv"
SORTIE = RACINE / "data" / "processed" / "black_friday_clean.csv"
DOSSIER_IMAGES = RACINE / "notebooks"

if not DATA.is_file():
    raise FileNotFoundError(f"Placer le fichier train.csv dans : {DATA.parent}")

df = pd.read_csv(DATA)
print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
df.head()


Dimensions : 550,068 lignes x 12 colonnes


,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
0,1000001,P00069042,F,0-17,10,A,2,0,3,NaN,NaN,8370
1,1000001,P00248942,F,0-17,10,A,2,0,1,6.0,14.0,15200
2,1000001,P00087842,F,0-17,10,A,2,0,12,NaN,NaN,1422
3,1000001,P00085442,F,0-17,10,A,2,0,12,14.0,NaN,1057
4,1000002,P00285442,M,55+,16,C,4+,0,8,NaN,NaN,7969


## 2. Contrôle qualité

Je vérifie les types, les valeurs manquantes, les doublons et les montants avant le nettoyage.

In [2]:
quality = pd.DataFrame({'type': df.dtypes.astype(str), 'manquants': df.isna().sum(), 'uniques': df.nunique()})
quality


,type,manquants,uniques
User_ID,int64,0,5891
Product_ID,object,0,3631
Gender,object,0,2
Age,object,0,7
Occupation,int64,0,21
City_Category,object,0,3
Stay_In_Current_City_Years,object,0,5
Marital_Status,int64,0,2
Product_Category_1,int64,0,20
Product_Category_2,float64,173638,17


In [3]:
print("Doublons complets :", df.duplicated().sum())
print("Montants manquants :", df['Purchase'].isna().sum())
print("Montants nuls ou négatifs :", df['Purchase'].le(0).sum())
print("Répartition des lignes par genre :")
print(df['Gender'].value_counts())


Doublons complets : 0
Montants manquants : 0
Montants nuls ou négatifs : 0
Répartition des lignes par genre :
Gender
M    414259
F    135809
Name: count, dtype: int64


Les catégories secondaires absentes restent nulles : leur remplacement par zéro créerait une catégorie artificielle. Je conserve les tranches d’âge comme catégories et ajoute des libellés français pour le genre et le statut matrimonial.

## 3. Nettoyage et export

Chaque ligne reçoit un identifiant technique `Transaction_ID`. Il identifie une ligne, pas un panier complet. Les identifiants client sont traités comme des libellés dans Python.

In [4]:
clean = df.drop_duplicates().copy()
clean.insert(0, 'Transaction_ID', range(1, len(clean) + 1))
clean['User_ID'] = clean['User_ID'].astype(str)
clean['Gender_Label'] = clean['Gender'].map({'M': 'Homme', 'F': 'Femme'})
clean['Marital_Status_Label'] = clean['Marital_Status'].map({0: 'Célibataire', 1: 'Marié(e)'})
clean['Stay_In_Current_City_Years'] = clean['Stay_In_Current_City_Years'].replace('4+', '4 ans ou plus')
for col in ['Product_Category_2', 'Product_Category_3']:
    clean[col] = clean[col].astype('Int64')
SORTIE.parent.mkdir(parents=True, exist_ok=True)
try:
    clean.to_csv(SORTIE, index=False)
except PermissionError as erreur:
    raise PermissionError("Fermer le CSV dans Excel et vérifier les droits d’écriture, puis relancer cette cellule.") from erreur
print(f'{len(clean):,} lignes propres exportées')


550,068 lignes propres exportées


In [5]:
print("Colonnes exportées :")
print(pd.read_csv(SORTIE, nrows=0).columns.tolist())


Colonnes exportées :
['Transaction_ID', 'User_ID', 'Product_ID', 'Gender', 'Age', 'Occupation', 'City_Category', 'Stay_In_Current_City_Years', 'Marital_Status', 'Product_Category_1', 'Product_Category_2', 'Product_Category_3', 'Purchase', 'Gender_Label', 'Marital_Status_Label']


## 4. Indicateurs clés

Je calcule le nombre de lignes d’achat, de clients et de produits distincts, puis le montant total, moyen et médian. Le jeu ne permet pas de reconstituer les paniers complets ; je parle donc de montant moyen par ligne d’achat. Aucune devise n’est ajoutée sans confirmation de la source.

In [6]:
kpi = {
    'Nombre de lignes d’achat': len(clean),
    'Nombre de clients': clean['User_ID'].nunique(),
    'Nombre de produits': clean['Product_ID'].nunique(),
    'Montant total des achats': clean['Purchase'].sum(),
    'Montant moyen d’un achat': clean['Purchase'].mean(),
    'Montant médian d’un achat': clean['Purchase'].median()
}
pd.Series(kpi)


Nombre de lignes d’achat     5.500680e+05
Nombre de clients            5.891000e+03
Nombre de produits           3.631000e+03
Montant total des achats     5.095813e+09
Montant moyen d’un achat     9.263969e+03
Montant médian d’un achat    8.047000e+03
dtype: float64

## 5. Analyse des clients et des produits

Pour chaque groupe, `count` compte les lignes, `sum` additionne les montants et `mean` calcule leur moyenne. Une somme élevée peut venir d’un grand nombre d’achats.

In [7]:
age_stats = clean.groupby('Age').Purchase.agg(['count','sum','mean']).sort_values('sum', ascending=False)
age_stats


,count,sum,mean
Age,,,
26-35,219587,2031770578,9252.690633
36-45,110013,1026569884,9331.350695
18-25,99660,913848675,9169.663606
46-50,45701,420843403,9208.625697
51-55,38501,367099644,9534.808031
55+,21504,200767375,9336.280459
0-17,15102,134913183,8933.464640


In [8]:
city_stats = clean.groupby('City_Category').Purchase.agg(['count','sum','mean']).sort_values('sum', ascending=False)
city_stats


,count,sum,mean
City_Category,,,
B,231173,2115533605,9151.300563
C,171175,1663807476,9719.920993
A,147720,1316471661,8911.939216


In [9]:
top_products = clean.groupby('Product_ID').Purchase.agg(['count','sum','mean']).nlargest(10, 'sum')
top_products


,count,sum,mean
Product_ID,,,
P00025442,1615,27995166,17334.468111
P00110742,1612,26722309,16577.114764
P00255842,1383,25168963,18198.816341
P00059442,1406,24338343,17310.343528
P00184942,1440,24334887,16899.227083
P00112142,1562,24216006,15503.204866
P00110942,1360,23639564,17382.032353
P00237542,1394,23425576,16804.573888
P00057642,1470,23102780,15716.176871


In [10]:
categorie_stats = clean.groupby('Product_Category_1')['Purchase'].agg(['count', 'sum', 'mean']).sort_values('sum', ascending=False)
categorie_stats


,count,sum,mean
Product_Category_1,,,
1,140378,1910013754,13606.218596
5,150933,941835229,6240.088178
8,113925,854318799,7498.958078
6,20466,324150302,15838.478550
2,23864,268516186,11251.935384
3,20213,204084713,10096.705734
16,9828,145120612,14766.037037
11,24287,113791115,4685.268456
10,5125,100837301,19675.570927


In [11]:
genre_stats = clean.groupby('Gender_Label')['Purchase'].agg(['count', 'sum', 'mean'])
genre_stats['part_montant_pct'] = genre_stats['sum'] / clean['Purchase'].sum() * 100
print(genre_stats.round(2))
part_26_35 = age_stats.loc['26-35', 'sum'] / clean['Purchase'].sum() * 100
print(f'Part du montant pour les 26–35 ans : {part_26_35:.1f} %')


               count         sum     mean  part_montant_pct
Gender_Label                                               
Femme         135809  1186232642  8734.57             23.28
Homme         414259  3909580100  9437.53             76.72
Part du montant pour les 26–35 ans : 39.9 %


### Observations et limites

- Les 26–35 ans représentent le montant total le plus élevé, avec un grand nombre de lignes d’achat.
- La catégorie de ville B domine en volume et en montant total ; la catégorie C présente le montant moyen le plus élevé. A, B et C sont des catégories, pas des noms de villes.
- Les catégories de produits 1, 5 et 8 sont les principales catégories par montant total.
- Les comparaisons par genre doivent tenir compte du nombre de lignes d’achat dans chaque groupe.
- Sans date, quantité, prix catalogue, remise ou marge, je ne peux pas mesurer l’effet causal des promotions ni la rentabilité.


## 6. Connexion Python à Azure SQL

Le CSV nettoyé est conservé dans Blob, puis importé dans `dbo.BlackFriday`. Les scripts sont utilisés dans cet ordre : `01_schema.sql`, `02_import_blob.sql`, `03_quality_checks.sql`, puis `04_views.sql` pour les vues complémentaires. Les trois premiers reprennent les commandes exportées depuis Azure ; les vues ne figurent pas dans ces exports et Power BI lit directement la table.

Cette partie vérifie la base existante avec une requête de comptage ; elle ne réimporte pas les données. Après un blocage de la connexion interactive, j’ai utilisé `DeviceCodeCredential` : le code affiché à chaque exécution permet de me connecter dans le navigateur. La sortie conservée confirme 550 068 lignes. Aucun jeton n’est enregistré.


In [30]:
import pyodbc
pilotes = pyodbc.drivers()
print(pilotes)
if "ODBC Driver 18 for SQL Server" not in pilotes:
    raise RuntimeError("Installer Microsoft ODBC Driver 18 for SQL Server sur Windows avant de continuer.")


['SQL Server', 'ODBC Driver 18 for SQL Server']


In [39]:
import pyodbc
import struct

# Connexion Microsoft dans le navigateur
from azure.identity import DeviceCodeCredential
identite = DeviceCodeCredential(timeout=300)
jeton = identite.get_token(
    "https://database.windows.net/.default"
).token

# Préparation du jeton pour le pilote ODBC
jeton_bytes = jeton.encode("utf-16-le")
jeton_odbc = struct.pack("<I", len(jeton_bytes)) + jeton_bytes

# Connexion à notre base Azure SQL
connexion = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=tcp:sql-blackfriday-241095.database.windows.net,1433;"
    "Database=db-blackfriday;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;",
    attrs_before={1256: jeton_odbc},
    timeout=120
)

curseur = connexion.cursor()
curseur.execute("SELECT COUNT(*) FROM dbo.BlackFriday")

print("Connexion à Azure SQL réussie.")
print("Nombre de lignes :", curseur.fetchone()[0])

Connexion à Azure SQL réussie.
Nombre de lignes : 550068


In [40]:
curseur.close()
connexion.close()
print("Connexion SQL fermée.")


Connexion SQL fermée.


## 7. Restitution dans Power BI

Le rapport Power BI utilise une seule table `BlackFriday` : aucune relation entre tables n’est nécessaire dans ce modèle. Les mesures calculent le nombre de lignes, le nombre de clients distincts, le montant total et le montant moyen des achats.

La page Vue d’ensemble présente les achats par âge, genre et catégorie de produits. Le filtre d’âge a été testé et modifie les indicateurs. La page Analyse détaillée présente une matrice produits × tranches d’âge avec un dégradé de couleur. Elle est limitée aux 10 premiers produits par montant total pour faciliter la lecture.

### Utilité professionnelle

Cette démarche peut servir à comparer les segments clients, repérer les produits importants et construire un tableau de bord de suivi. Les exemples d’IA illustrent la rédaction de descriptions, la synthèse d’avis et l’identification visuelle de produits. Des données supplémentaires seraient nécessaires pour prévoir les ventes, analyser les marges ou évaluer l’efficacité d’une promotion.



## 8. Azure OpenAI : description et résumé

Les produits et avis de cette démonstration sont fictifs et ne proviennent pas du CSV Black Friday. Le but est de montrer une utilisation de l’IA dans le commerce : rédiger une description à partir de caractéristiques fournies et résumer un avis.

Les caractéristiques du casque fictif ne décrivent pas nécessairement le casque de la photographie utilisée plus loin. Les clés ne sont pas enregistrées dans le code.


In [18]:
from openai import OpenAI
from getpass import getpass

api_key = getpass("Colle ta clé API Azure OpenAI ici :")

In [19]:
client = OpenAI(
    base_url="https://aoai-blackfriday-241095.openai.azure.com/openai/v1/",
    api_key=api_key
)

# Produit fictif pour la démonstration
prompt = (
    "Rédige une description marketing en français de 3 phrases maximum "
    "pour un casque Bluetooth noir avec réduction de bruit et autonomie "
    "de 30 heures. N'invente aucune caractéristique ni promotion."
)

reponse = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": prompt}]
)

description = reponse.choices[0].message.content
print(description)

Découvrez notre casque Bluetooth noir alliant confort et performance, équipé d’une technologie avancée de réduction de bruit pour une écoute immersive. Profitez d’une autonomie exceptionnelle de 30 heures pour accompagner tous vos moments. Un design élégant et pratique pour une expérience audio sans compromis.


In [20]:
# Avis client fictif pour la démonstration
avis = (
    "J'ai acheté ce casque pendant le Black Friday. "
    "La réduction de bruit est efficace et la batterie tient longtemps. "
    "En revanche, il devient inconfortable après plusieurs heures "
    "et la livraison a pris du retard."
)

reponse_avis = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{
        "role": "user",
        "content": (
            "Résume cet avis client en deux phrases maximum, "
            "sans ajouter d'information : " + avis
        )
    }]
)

resume_avis = reponse_avis.choices[0].message.content
print(resume_avis)


Le casque offre une bonne réduction de bruit et une batterie durable, mais devient inconfortable après plusieurs heures d'utilisation. De plus, la livraison a été retardée.


## 9. Azure Language : sentiments et expressions clés

Je classe trois avis fictifs en positif, négatif ou neutre, puis j’extrais leurs expressions importantes. Les scores représentent la confiance du service, pas une garantie d’exactitude.

In [21]:
from getpass import getpass
language_key = getpass("colle la clé Azure Language ici : ")


In [22]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

language_client = TextAnalyticsClient(
    endpoint="https://lang-blackfriday-241095.cognitiveservices.azure.com/",
    credential=AzureKeyCredential(language_key)
)

# Avis fictifs pour la démonstration
avis_clients = [
    "Je suis très satisfait du casque. Le son est excellent.",
    "Le produit est défectueux et la livraison était en retard.",
    "Le casque est noir et possède une connexion Bluetooth."
]

resultats_sentiments = language_client.analyze_sentiment(
    documents=avis_clients,
    language="fr"
)

for avis, resultat in zip(avis_clients, resultats_sentiments):
    print("Avis :", avis)
    if resultat.is_error:
        print("Erreur :", resultat.error.message)
    else:
        print("Sentiment :", resultat.sentiment)
    print()

Avis : Je suis très satisfait du casque. Le son est excellent.
Sentiment : positive

Avis : Le produit est défectueux et la livraison était en retard.
Sentiment : negative

Avis : Le casque est noir et possède une connexion Bluetooth.
Sentiment : neutral



In [23]:
for avis, resultat in zip(avis_clients, resultats_sentiments):
    if not resultat.is_error:
        print("Avis :", avis)
        print("Scores :", resultat.confidence_scores)
        print()


Avis : Je suis très satisfait du casque. Le son est excellent.
Scores : {'positive': 1.0, 'neutral': 0.0, 'negative': 0.0}

Avis : Le produit est défectueux et la livraison était en retard.
Scores : {'positive': 0.0, 'neutral': 0.0, 'negative': 1.0}

Avis : Le casque est noir et possède une connexion Bluetooth.
Scores : {'positive': 0.0, 'neutral': 1.0, 'negative': 0.0}



In [24]:
resultats_cles = language_client.extract_key_phrases(
    documents=avis_clients,
    language="fr"
)

for avis, resultat in zip(avis_clients, resultats_cles):
    print("Avis :", avis)
    if resultat.is_error:
        print("Erreur :", resultat.error.message)
    else:
        print("Expressions clés :", ", ".join(resultat.key_phrases))
    print()

Avis : Je suis très satisfait du casque. Le son est excellent.
Expressions clés : casque, son

Avis : Le produit est défectueux et la livraison était en retard.
Expressions clés : produit, livraison, retard

Avis : Le casque est noir et possède une connexion Bluetooth.
Expressions clés : casque, connexion, Bluetooth



## 10. Azure Computer Vision : analyse d’une image

Une image de casque, indépendante du CSV, sert à obtenir une description et des tags avec leurs scores. Les sorties sont demandées en anglais comme dans notre démonstration. Le fichier image doit accompagner le notebook ; Conserver aussi son URL d’origine pour identifier la photographie utilisée.

In [25]:
from getpass import getpass
vision_key = getpass("colle la clé Azure Vision ici : ")

In [26]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from msrest.authentication import CognitiveServicesCredentials

vision_endpoint = "https://vision-blackfriday-241095.cognitiveservices.azure.com/"

vision_client = ComputerVisionClient(
    vision_endpoint,
    CognitiveServicesCredentials(vision_key)
)

print("Client Azure Vision prêt.")

Client Azure Vision prêt.


In [27]:
from PIL import Image

IMAGE_JPG = DOSSIER_IMAGES / "produit.jpg"
IMAGE_WEBP = DOSSIER_IMAGES / "produit.webp"
if not IMAGE_JPG.is_file():
    if not IMAGE_WEBP.is_file():
        raise FileNotFoundError("Ajouter produit.jpg ou produit.webp dans le dossier notebooks.")
    with Image.open(IMAGE_WEBP) as image:
        image.convert("RGB").save(IMAGE_JPG, "JPEG")


In [28]:
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes

with open(IMAGE_JPG, "rb") as image:
    resultat_vision = vision_client.analyze_image_in_stream(
        image,
        visual_features=[
            VisualFeatureTypes.description,
            VisualFeatureTypes.tags
        ],
        language="en"
    )

print("Description :")
for description in resultat_vision.description.captions:
    print(description.text)

print("\nMots clés :")
for tag in resultat_vision.tags:
    print(tag.name)


Description :
a pair of headphones

Mots clés :
electronics
headphones
earphone
audio equipment
headset
electronic device
music
black and white
headphone


In [29]:
for tag in resultat_vision.tags:
    print(f"{tag.name} : {tag.confidence:.1%}")


electronics : 99.5%
headphones : 98.4%
earphone : 97.9%
audio equipment : 88.3%
headset : 87.4%
electronic device : 85.4%
music : 58.7%
black and white : 57.8%
headphone : 48.2%
